# Experiment 028 (W14): GRPO after MC-SFT (Colab)

Three arms at 7B, seeds 0 to 2, each starting grpo from its seed-matched 014
mc-sft adapter (base to SFT(MMLU) to GRPO(MMLU)). The registered rules from the
W14 plan v4 apply: adapter inventory preflight with hash bindings before any
contact, parent identity probes in disposable subprocesses, 70 GiB GPU and 70GB
Drive gates, base evals prepopulated from 013 and never recomputed, training
through the disposable worker with the merge bound to its source, and the
standard resume-safe evaluation machinery writing only into the 028 directories.

Seed 0 runs first as the technical pilot for the merge path and memory. After a
clean seed 0, set the seed list to all three and Run-all again; finished work
skips.

## 0. Compute units and mount Drive

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 682.45     # fill from the resources panel
unit_rate = 6.77     # units per hour for this card
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '028_colab_mcsft_grpo',
                 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f)
        print('opened session record')

session 20260811_191335 | cu_before = 682.45
Mounted at /content/drive
closed previous session: 20260811_121007
opened session record


## 1. GPU check: A100 with 80GB required

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime, the 7b cell card per rule 6z'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-bbaa3e35-9205-6967-a2d1-c08fb7b3c22b)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3637, done.
remote: Counting objects: 100% (1121/1121), done.
remote: Compressing objects: 100% (779/779), done.
remote: Total 3637 (delta 672), reused 779 (delta 340), pack-reused 2516 (from 1)
Receiving objects: 100% (3637/3637), 112.02 MiB | 31.07 MiB/s, done.
Resolving deltas: 100% (2291/2291), done.
Updating files: 100% (1659/1659), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 158.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.7 MB/s eta 0:00:00


## 4. W&B key (automatic)

In [5]:
import getpass, os
key = None
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
except Exception:
    key = None
if not key and os.path.exists('wandb_api_key.txt'):
    key = open('wandb_api_key.txt').read().strip()
if not key:
    key = getpass.getpass('wandb api key: ').strip()
assert key, 'a wandb key is required, the worker logs online and must not wait for a login'
os.environ['WANDB_API_KEY'] = key
with open('wandb_api_key.txt', 'w') as f:
    f.write(key)
key = None
print('wandb key set')

wandb key set


## 5. Arm selection: seed 0 pilots the merge path

In [6]:
# seeds 0 and 1 both ended in sentinel aborts and stay blocked per the
# no-mutation rule, so only the last fresh arm runs here
SEEDS_TO_RUN = [2]
print('arms this session:', SEEDS_TO_RUN)


arms this session: [2]


## 6. Preflight: source adapters, parents, capacity, before any contact

In [7]:
import json, os, shutil, sys
sys.path.insert(0, os.getcwd())
from config import load_config, run_tag
from scripts.s12_eval import atomic_write_json, config_hash_status
from scripts.s28_compose import ARMS_028, resolve_source_adapter

D = '/content/drive/MyDrive/inm363'
ADIR, PREFLIGHT, TAGS = {}, {}, {}
problems = []
for cfg_path, seed, parent_tag in ARMS_028:
    cfg = load_config(cfg_path)
    assert cfg.train.seed == seed
    TAGS[seed] = run_tag(cfg)
    try:
        adir, hashes = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
    except Exception as e:
        problems.append(f'seed {seed}: {e}')
        continue
    ADIR[seed], PREFLIGHT[seed] = adir, hashes

    # the stored hash must belong to this exact 014 parent, current or 014-era
    parent_cfg = load_config('configs/014_p8_colab_b7.yaml')
    parent_cfg.train.seed = seed
    parent_man = json.load(open(os.path.join(D, 'results', parent_tag, 'manifest.json')))
    stored = open(os.path.join(adir, 'config_hash.txt')).read().strip()
    status = config_hash_status(parent_cfg, parent_man, stored)
    if status not in ('current', 'legacy_missing_grpo_from_adapter'):
        problems.append(f'seed {seed}: adapter hash status {status}')
        continue

    # the parent artifacts the gates and the summary reference need
    pdir = os.path.join(D, 'results', parent_tag)
    for n in ('sft_logprob_preds.jsonl', 'sft_logprob.json', 'sft_eval.json', 'sft_preds.jsonl'):
        if not os.path.exists(os.path.join(pdir, n)):
            problems.append(f'seed {seed}: parent missing {n}')
    print(f'ok: seed {seed} <- {parent_tag} ({status})')

free_gb = shutil.disk_usage(D).free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
if free_gb < 70:
    problems.append(f'only {free_gb:.0f}GB free on drive, three merged 7b bases need 70')

assert not problems, 'preflight halt:\n' + '\n'.join(problems)

# the hashes persist on drive; a rerun must see the same adapters, not trust memory
pf_path = os.path.join(D, 'results', '028_preflight.json')
record = {str(s): PREFLIGHT[s] for s in PREFLIGHT}
if os.path.exists(pf_path):
    prev = json.load(open(pf_path))
    assert prev == record, 'source adapters changed since the recorded preflight, halting'
    print('preflight matches the recorded hashes')
else:
    atomic_write_json(pf_path, record)
    print('preflight recorded')
print('preflight green for all three arms')

ok: seed 0 <- 014_p8_COLAB_FULL_B7_sft_only_seed0 (legacy_missing_grpo_from_adapter)
ok: seed 1 <- 014_p8_COLAB_FULL_B7_sft_only_seed1 (legacy_missing_grpo_from_adapter)
ok: seed 2 <- 014_p8_COLAB_FULL_B7_sft_only_seed2 (legacy_missing_grpo_from_adapter)
drive free: 178 GB
preflight matches the recorded hashes
preflight green for all three arms


## 7. Build the data splits (after the preflight)

In [8]:
!python scripts/build_data.py

README.md: 100% 7.93k/7.93k [00:00<00:00, 23.9MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 2.59MB/s,  226kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 2.60MB/s,  227kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:  12% 50.8k/419k [00:00<00:05, 68.8kB/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 536kB/s, 41.3kB/s  ] 
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 536kB/s, 41.3kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 402315.96 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 293739.35 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 92.0MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 123MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:  88% 81.9k/92.8k [

## 8. Base evals: prepopulated from 013, never recomputed

In [9]:
from scripts.s28_compose import prepopulate_base_evals
src = os.path.join(D, 'results', 'base_evals', '013_COLAB_FULL_B7')
dst = os.path.join(D, 'results', 'base_evals', '028_COLAB_FULL_B7')
rec = prepopulate_base_evals(src, dst)
print('reuse record covers', sorted(rec['sha256']))

  base evals already prepopulated and verified
reuse record covers ['base_eval.json', 'base_logprob.json', 'base_logprob_preds.jsonl', 'base_preds.jsonl']


## 9. Wiring check: the package tests run inside the session

In [10]:
# the package tests run inside the session and a failure stops everything here
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest',
                    'tests/test_s28_composition.py', 'tests/test_checkpoint.py', '-q'])
assert r.returncode == 0, 'package tests failed, nothing runs'

from scripts.grpo_worker import (active_worker_processes, blocked_resume_reason,
                                 stream_subprocess)
from scripts.s12_eval import GateFailure, IntegrityError, identity_check, read_jsonl, rows_aligned, sha256
from scripts.s28_compose import (arm_state, commit_endpoint_eval, endpoint_state,
                                 gate_state, verify_binding, write_arm_complete)
from obs import _versions, append_ledger, gpu_snapshot, _git_info
from train.model_utils import resolve_dtype
print('imports fine | tags:', [TAGS[s] for s in TAGS])

imports fine | tags: ['028_COLAB_FULL_B7_mcsft_grpo_seed0', '028_COLAB_FULL_B7_mcsft_grpo_seed1', '028_COLAB_FULL_B7_mcsft_grpo_seed2']


In [11]:
!pip uninstall -q -y torchao

## 10. Parent identity probes, each in a disposable subprocess

In [12]:
# all three parent identity probes run before any training, whatever the arm
# selection, so the package contact stays fixed at nine passes. Each probe runs
# in a child process that dies and releases its memory, and each completed gate
# persists on drive so a runtime restart never repeats the contact
import shutil as _shutil, subprocess, sys, time
from scripts.s12_eval import atomic_write_json

GATE_SCRIPT = '/content/s28_gate.py'
with open(GATE_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
run_logprob_eval(cfg, resolve_device(cfg.env), adapter_path=args.adapter,
                 condition="gate", out_dir=args.out)
''')

for cfg_path, seed, parent_tag in ARMS_028:
    gate_path = os.path.join(D, 'results', TAGS[seed], 'parent_gate.json')
    os.makedirs(os.path.dirname(gate_path), exist_ok=True)
    archived_p = os.path.join(D, 'results', parent_tag, 'sft_logprob_preds.jsonl')
    if gate_state(gate_path, PREFLIGHT[seed], archived_p) == 'committed':
        print(f'seed {seed}: gate already committed, skipping')
        continue

    gate_dir = f'/content/s28_gate_seed{seed}'
    gate_preds = os.path.join(gate_dir, 'gate_logprob_preds.jsonl')
    # an incomplete scratch from a dead runtime is rebuilt, never trusted
    rows_ok = (os.path.exists(gate_preds)
               and sum(1 for l in open(gate_preds) if l.strip()) == 1884)
    if not rows_ok:
        _shutil.rmtree(gate_dir, ignore_errors=True)
        t0 = time.time()
        r = subprocess.run([sys.executable, GATE_SCRIPT, '--config', cfg_path,
                            '--adapter', ADIR[seed], '--out', gate_dir])
        assert r.returncode == 0, f'seed {seed}: gate probe subprocess failed'
        elapsed = round(time.time() - t0, 1)
    else:
        elapsed = None
    new_rows = read_jsonl(gate_preds)
    assert len(new_rows) == 1884, f'seed {seed}: gate scratch is short'
    stats = identity_check(new_rows, read_jsonl(archived_p))
    atomic_write_json(gate_path, {
        'ts': time.strftime('%Y-%m-%d %H:%M'), 'completed': True,
        'parent': parent_tag, 'adapter_sha256': PREFLIGHT[seed],
        'archived_preds_sha256': sha256(archived_p),
        'new_preds_sha256': sha256(gate_preds),
        'stats': stats, 'elapsed_sec': elapsed})
    _shutil.rmtree(gate_dir, ignore_errors=True)
    print(f'seed {seed} gate: {stats}')

import torch
free_gib = torch.cuda.mem_get_info()[0] / 1024**3
print(f'gpu free: {free_gib:.1f} GiB')
assert free_gib >= 70, 'need 70 GiB free before the 7b training run'

seed 0: gate already committed, skipping
seed 1: gate already committed, skipping
seed 2: gate already committed, skipping
gpu free: 78.8 GiB


## 11. The run: bound merge, worker training, standard evaluation

In [ ]:
# both cost fields must be filled before the long runs start
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
import glob as _glob, subprocess, sys, time
from datetime import datetime

EVAL_SCRIPT = '/content/s28_eval.py'
with open(EVAL_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from checkpoint import save_metrics
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_eval, run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
device = resolve_device(cfg.env)
ev = run_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_eval.json"), ev)
lp = run_logprob_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_logprob.json"), lp)
''')

os.environ.setdefault('WANDB_PROJECT', 'inm363-grpo')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
LEDGER = os.path.join(D, 'results', 'compute_ledger.jsonl')
allowed_infra_files = {
    'notebooks/028_colab_mcsft_grpo.ipynb',
    'scripts/grpo_worker.py',
    'tests/test_s28_composition.py',
    'tests/test_grpo_worker.py',
}

for cfg_path, seed, parent_tag in ARMS_028:
    if seed not in SEEDS_TO_RUN:
        continue
    tag = TAGS[seed]
    print('===== ' + tag + ' =====')
    cfg = load_config(cfg_path)

    # each arm checks against the environments both parents recorded, dataset
    # revision pins included where the manifest carries them
    for man_tag in (parent_tag, f'013_COLAB_FULL_B7_sft_grpo_seed{seed}'):
        parent = json.load(open(os.path.join(D, 'results', man_tag, 'manifest.json')))
        for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate',
                    'cais/mmlu@', 'openai/gsm8k@'):
            want = parent.get('versions', {}).get(lib)
            if want is not None:
                got = _versions().get(lib)
                assert got == want, f'{man_tag} {lib}: {got} vs {want}'
        if parent.get('resolved_dtype'):
            assert resolve_dtype(cfg) == parent['resolved_dtype'], f'{man_tag}: dtype differs'
    assert cfg.exec.allow_oom_fallback is False, 'the ladder must stay off'
    assert cfg.log.wandb_project == 'inm363-grpo', 'wandb project comes from the config'

    out_dir = os.path.join(D, 'checkpoints', tag, 'grpo')
    merged_dir = os.path.join(D, 'checkpoints', tag, 'sft_merged')
    res_dir = os.path.join(D, 'results', tag)
    os.makedirs(res_dir, exist_ok=True)
    final_artifacts = [os.path.join(out_dir, 'adapter_model.safetensors'),
                       os.path.join(out_dir, 'adapter_base.json'),
                       os.path.join(res_dir, 'grpo_trainlog.json')]

    # fully done only when the completion record verifies the summary, the
    # manifest and the adapter-bound endpoint; anything inconsistent halts
    if arm_state(res_dir, out_dir) == 'committed':
        verify_binding(ADIR[seed], merged_dir)
        print('  arm complete, skipping')
        continue

    man_path = os.path.join(res_dir, 'arm_manifest.json')
    man = json.load(open(man_path)) if os.path.exists(man_path) else {'attempts': []}

    def newest_step():
        cks = _glob.glob(os.path.join(out_dir, 'checkpoint-*'))
        return max((int(c.rsplit('-', 1)[1]) for c in cks), default=0)

    start_step = newest_step()
    for a in man.get('attempts', []):
        if 'ts_end' not in a:
            a.update({'interrupted': True, 'end_step': start_step,
                      'elapsed_sec': None, 'ts_end': 'runtime died before finally'})
    with open(man_path, 'w') as f:
        json.dump(man, f, indent=2)

    reason = blocked_resume_reason(man.get('attempts', []))
    assert reason is None, f'seed {seed}: {reason}'
    assert not active_worker_processes(cfg_path), 'a training worker is already running'

    trained = all(os.path.exists(f) for f in final_artifacts)
    if not trained:
        # code drift between attempts is only tolerated for the named
        # infrastructure files, the 022 rule
        env = {'git': _git_info(), 'versions': _versions(),
               'resolved_dtype': resolve_dtype(cfg), 'ladder': 'disabled'}
        repo_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True,
                                  text=True, check=True).stdout.strip()
        repo_prefix = os.path.relpath(os.getcwd(), repo_top)
        for a in man.get('attempts', []):
            prev = a.get('env')
            if prev:
                old_c, new_c = prev['git'].get('commit'), env['git'].get('commit')
                if old_c != new_c:
                    diff = subprocess.run(['git', 'diff', '--name-only', old_c, new_c],
                                          capture_output=True, text=True, check=True).stdout.splitlines()
                    changed = []
                    for name in diff:
                        prefix = repo_prefix + os.sep
                        changed.append(name[len(prefix):] if repo_prefix != '.' and name.startswith(prefix) else name)
                    unexpected = sorted(set(changed) - allowed_infra_files)
                    assert not unexpected, f'seed {seed}: scientific code drifted between attempts: {unexpected}'
                for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
                    assert prev['versions'].get(lib) == env['versions'].get(lib), f'seed {seed}: {lib} drifted'
                assert prev['resolved_dtype'] == env['resolved_dtype'], f'seed {seed}: dtype drifted between attempts'

        free_gib = torch.cuda.mem_get_info()[0] / 1024**3
        assert free_gib >= 70, f'only {free_gib:.1f} GiB free before training'

        attempt = {'start_step': start_step, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                   'gpu': gpu_snapshot(), 'env': env,
                   'adapter_sha256': PREFLIGHT[seed], 'completed': False}
        man['attempts'].append(attempt)
        with open(man_path, 'w') as f:
            json.dump(man, f, indent=2)

        status_path = f'/content/028_worker_seed{seed}.json'
        if os.path.exists(status_path):
            os.remove(status_path)
        worker_cmd = [sys.executable, '-u', 'scripts/grpo_worker.py',
                      '--config', cfg_path, '--adapter', ADIR[seed],
                      '--out-dir', out_dir, '--log-dir', res_dir,
                      '--run-name', f'{tag}_grpo', '--status', status_path,
                      '--report-to', 'wandb']
        t0 = time.time()
        proc = None
        parent_failure = None
        try:
            print(f'===== seed {seed}, from step {start_step} =====')
            proc = stream_subprocess(worker_cmd)
            if proc.returncode != 0:
                raise RuntimeError(f'seed {seed}: training worker failed with exit code {proc.returncode}')
        except BaseException as exc:
            parent_failure = {'error_type': type(exc).__name__,
                              'error_message': str(exc)[:2000]}
            raise
        finally:
            wall = time.time() - t0
            end_step = newest_step()
            worker_returncode = proc.returncode if proc is not None else None
            worker_status = json.load(open(status_path)) if os.path.exists(status_path) else {}
            worker_peak = worker_status.get('peak_vram_gib')
            error_type = worker_status.get('error_type') or (parent_failure or {}).get('error_type')
            error_message = worker_status.get('error_message') or (parent_failure or {}).get('error_message')
            manual_interrupt = error_type == 'KeyboardInterrupt'
            explicit_failure = ((worker_returncode not in (None, 0) or parent_failure is not None)
                                and not manual_interrupt)
            done = (worker_status.get('completed') is True
                    and all(os.path.exists(f) for f in final_artifacts))
            attempt.update({'end_step': end_step, 'ts_end': time.strftime('%Y-%m-%d %H:%M'),
                            'elapsed_sec': round(wall, 1), 'completed': done,
                            'peak_vram_gib': worker_peak, 'worker_returncode': worker_returncode,
                            'error_type': error_type, 'error_message': error_message,
                            'cuda_oom': bool(worker_status.get('cuda_oom', False)),
                            'interrupted': manual_interrupt, 'resume_blocked': explicit_failure})
            with open(man_path, 'w') as f:
                json.dump(man, f, indent=2)
            gpu_row = gpu_snapshot()
            if worker_peak is not None:
                gpu_row['peak_vram_gb'] = worker_peak
            append_ledger(LEDGER, {'run_tag': tag, 'condition': 'MCSFT_GRPO_train',
                                   'timings_sec': {'grpo_train': round(wall, 1)},
                                   'train_steps': end_step - start_step,
                                   'completed': attempt['completed'], 'gpu': gpu_row})
        assert worker_status.get('completed') is True, f'seed {seed}: worker status did not complete'
        assert attempt['completed'] is True, f'seed {seed}: final training artifacts are incomplete'
        print(f'  trained in {wall/3600:.2f}h, peak {worker_peak} GiB')
    else:
        print('  adapter already trained')

    # the trained-from weights re-verify before any evaluation, every time
    verify_binding(ADIR[seed], merged_dir)
    _, hashes_now = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
    assert hashes_now == PREFLIGHT[seed], f'seed {seed}: source adapter drifted since preflight'

    # Transactional endpoint evaluation: subprocess into scratch, then commit.
    # A tampered endpoint halts here, before any new test-set contact
    ep_state = endpoint_state(res_dir, out_dir)
    if ep_state == 'tampered':
        raise IntegrityError(f'seed {seed}: endpoint files contradict their manifest, '
                             'halting before any re-evaluation')
    if ep_state != 'committed':
        sdir = f'/content/s28_eval_seed{seed}'
        _shutil.rmtree(sdir, ignore_errors=True)
        t0 = time.time()
        eval_done = False
        try:
            r = subprocess.run([sys.executable, EVAL_SCRIPT, '--config', cfg_path,
                                '--adapter', out_dir, '--out', sdir])
            assert r.returncode == 0, f'seed {seed}: evaluation subprocess failed'
            eval_wall = round(time.time() - t0, 1)
            gate_rec = json.load(open(os.path.join(res_dir, 'parent_gate.json')))
            extra = {'adapter': out_dir, 'source_adapter': ADIR[seed],
                     'source_adapter_sha256': PREFLIGHT[seed],
                     'merged_binding': verify_binding(ADIR[seed], merged_dir),
                     'parent_gate_stats': gate_rec['stats'],
                     'config': cfg_path, 'seed': seed, 'git': _git_info(),
                     'versions': _versions(), 'resolved_dtype': resolve_dtype(cfg),
                     'gpu': gpu_snapshot(), 'timings_sec': {'grpo_eval': eval_wall}}
            commit_endpoint_eval(res_dir, sdir, adapter_dir=out_dir, extra_manifest=extra)
            _shutil.rmtree(sdir, ignore_errors=True)
            eval_done = True
        finally:
            append_ledger(LEDGER, {'run_tag': tag, 'condition': 'MCSFT_GRPO_eval',
                                   'timings_sec': {'grpo_eval': round(time.time() - t0, 1)},
                                   'train_steps': 0, 'completed': eval_done,
                                   'gpu': gpu_snapshot()})

    # summary and manifest through proven runner: training, base evals and
    # the committed endpoint files all skip inside it
    r = subprocess.run([sys.executable, 'scripts/run.py', '--config', cfg_path,
                        '--stage', 'all', '--no-ledger'])
    assert r.returncode == 0, f'seed {seed}: summary stage failed'
    s = json.load(open(os.path.join(res_dir, 'summary.json')))
    assert s['reference'].startswith('mcsft_parent:'), 'summary must reference the 014 parent'
    ev = json.load(open(os.path.join(res_dir, 'grpo_eval.json')))
    write_arm_complete(res_dir)
    print('  gen per-tier:', ev['per_tier_acc'])

print('session done for arms', SEEDS_TO_RUN)

Streaming output truncated to the last 5000 lines.
│ │ these         │               │               │              │           │ │
│ │ products are  │               │               │              │           │ │
│ │ stocked?      │               │               │              │           │ │
│ │ A. Basic      │               │               │              │           │ │
│ │ stock list    │               │               │              │           │ │
│ │ B. Buying     │               │               │              │           │ │
│ │ plan          │               │               │              │           │ │
│ │ C.            │               │               │              │           │ │
│ │ Merchandise   │               │               │              │           │ │
│ │ mix           │               │               ��              │           │ │
│ │ D. Price-line │               │               │              │           │ │
│ │ list          │               │               │      

## 12. Peek at the results

In [ ]:
for cfg_path, seed, parent_tag in ARMS_028:
    res_dir = os.path.join(D, 'results', TAGS[seed])
    sp = os.path.join(res_dir, 'summary.json')
    if not os.path.exists(sp):
        print(TAGS[seed], ': not run yet')
        continue
    s = json.load(open(sp))
    print(TAGS[seed])
    print('  reference       :', s['reference'])
    print('  parent mc-sft   :', s['sft_acc'])
    print('  composition grpo:', s['grpo_acc'])
    print('  grpo delta      :', s['grpo_delta'])
    print('  probe delta     :', s['grpo_logprob_delta'])

028_COLAB_FULL_B7_mcsft_grpo_seed0 : not run yet
028_COLAB_FULL_B7_mcsft_grpo_seed1 : not run yet
028_COLAB_FULL_B7_mcsft_grpo_seed2
  reference       : mcsft_parent:014_p8_COLAB_FULL_B7_sft_only_seed2
  parent mc-sft   : {'easy': 0.8907849829351536, 'medium': 0.7783251231527094, 'hard': 0.5580808080808081}
  composition grpo: {'easy': 0.8873720136518771, 'medium': 0.7783251231527094, 'hard': 0.5580808080808081}
  grpo delta      : {'easy': -0.0034129692832765013, 'medium': 0.0, 'hard': 0.0}
  probe delta     : {'easy': 0.0022753128555176305, 'medium': -0.0016420361247947435, 'hard': -0.002525252525252597}


## 13. Close the units record

In [ ]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 114.5      # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

logged: {'session_id': '20260811_191335', 'ts_start': '2026-08-11 19:13', 'notebook': '028_colab_mcsft_grpo', 'colab_cu_before': 682.45, 'unit_rate': 6.77, 'ts_end': '2026-08-11 19:54', 'colab_cu_after': 114.5, 'colab_cu_used': 567.95, 'closed_on_resume': False}
